In [ ]:
import pandas as pd
import os
import kagglehub
import zipfile

In [ ]:
def run_vietnam_rent_etl():
    print("Extract dataset from Kaggle API...")
    try:
        download_path = kagglehub.dataset_download(
            "cresht2606/vietnam-real-estate-datasets-catalyst")
        print(f"Dataset downloaded to: {download_path}")

        for item in os.listdir(download_path):
            if item.endswith('.zip'):
                zip_ref = zipfile.ZipFile(
                    os.path.join(download_path, item), 'r')
                print(f"Unzipping {item}...")
                zip_ref.extractall(download_path)
                zip_ref.close()
        
        target_file = os.path.join(
            download_path, "house_rental_dec29th_2025.csv")
        
        if not os.path.exists(target_file):
            print(f'Could not find house_rental_dec29th_2025.csv in {download_path}')
            return None
        
        df = pd.read_csv(target_file)
        print(f"Successfully loaded {len(df)} rows.")

        return df
    except Exception as e:
        print(f"Extraction failed: {e}")
        return

In [ ]:
if __name__ == "__main__":
    run_vietnam_rent_etl()
    df_raw = run_vietnam_rent_etl()

In [ ]:
# We can see that the location has both the district and the city. And some have the city and the province.
print(df_raw.head())

In [ ]:
# We can also see that there are 4808 id's, but the area, bedrooms, bathrooms, floors, price_million_vnd are all having less than that.
print(df_raw.info())

In [ ]:
import numpy as np

In [ ]:
def transform_vietnam_rent_data(df_raw):
    print("Transforming...")

    df_clean = df_raw.copy()

    # Drop the rows that have missing important values
    df_clean.dropna(subset=['area_m2', 'price_million_vnd'], inplace=True)
    rows_dropped = len(df_raw) - len(df_clean)
    print(f'{rows_dropped} rows dropped because of missing data in area and price.')

    # Since the number of missing values from bedrooms, bathrooms, floors are too many, and it usually just mean that it's a studio,...
    # So I will replace them with a "1" and as integer, since there are no 1.5 bed room.
    df_clean['bedrooms'] = df_clean['bedrooms'].fillna(1).astype(int)
    df_clean['bathrooms'] = df_clean['bathrooms'].fillna(1).astype(int)
    df_clean['floors'] = df_clean['floors'].fillna(1).astype(int)
    print('Replace the missing bedrooms, bathrooms, and floors with 1.')

    # I will seperate the location into 2, but there are cities inside provinces and district inside cities, I will divide them into
    # 2 levels instead of just cities and districts
    df_clean[['location_level_2', 'location_level_1']] = df_clean['location'].str.split(', ', n=1, expand=True)
    print('Successfully splitted location into 2 parts.')

    # Calculate millions into the raw number, and then divide by area to find the KPI
    df_clean = df_clean[(df_clean['price_million_vnd'] > 0)
                        & (df_clean['area_m2'] > 0)]
    df_clean['price_per_m2_vnd'] = ((df_clean['price_million_vnd']*1000000)/df_clean['area_m2']).round(0)

    # Some property have outlier price, so we also remove that by choosing only from the middle 50% of both
    Q1 = df_clean['price_per_m2_vnd'].quantile(0.25)
    Q3 = df_clean['price_per_m2_vnd'].quantile(0.75)
    IQR = Q3 - Q1
    # After checking, the data seems to be right skewed, so the lower bound calculated by IQR will negative, so I just give a ceiling
    lower_bound = 30000
    upper_bound = Q3 + 1.5 * IQR
    # Keep only the rows inside
    df_clean = df_clean[(df_clean['price_per_m2_vnd'] >= lower_bound) & (
        df_clean['price_per_m2_vnd'] <= upper_bound)]

    # Since this data seems to have some industrial area too, I will put a ceiling for the area too
    # 1000 m2 is a safe cutoff for residential/standard commercial
    df_clean = df_clean[df_clean['area_m2'] <= 1000]

    # Since there are properties that are posted by different people, I will deal with it by removing the ones that have the same
    # location, area, bedrooms, price.
    df_clean = df_clean.drop_duplicates(
        subset=['location_level_2', 'area_m2', 'bedrooms', 'price_million_vnd'], keep='first')
    print(f"Final row count: {len(df_clean)}")

    print(f'Transformation completed. Total of {len(df_clean)} rows ready.')

    return df_clean

In [ ]:
if __name__ == "__main__":
    if df_raw is not None:
        df_transformed = transform_vietnam_rent_data(df_raw)
        print("Preview cleaned data...")
        print(df_transformed.head())

In [ ]:
import sqlite3

In [ ]:
def load_vietnam_rent_data(df_clean, db_name="vietnam_rentals.db"):
    print("Loading data...")

    # Create the db file
    # conn = sqlite3.connect(db_name)
    # # Load data to table 'listings'
    # df_clean.to_sql('listings', conn, if_exists='replace', index=False)
    # conn.close()

    df_clean.to_csv('vietnam_rentals_powerbi.csv', index=False)
    print(f"Data successfully saved as csv.")

In [ ]:
if __name__ == "__main__":
    # Extract
    df_raw = run_vietnam_rent_etl()

    if df_raw is not None:
        # Transform
        df_clean = transform_vietnam_rent_data(df_raw)

        # Load
        load_vietnam_rent_data(df_clean)